# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset describing clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata and schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset loaded!\nName: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review and print available record sets, fields, and their `@id`s. Each entity—including record sets and fields—has a unique `@id` according to the Croissant specification.

**Note:** Use the `.record_sets` property of `dataset.metadata` to programmatically list the available record sets and their fields.

In [ ]:
# List all record sets and their fields, referencing `@id`
print("Available record sets and fields (with their @id):\n")
record_sets = getattr(metadata, 'record_sets', [])
for rs in record_sets:
    print(f"- RecordSet name: {getattr(rs, 'name', 'no name')} | @id: {getattr(rs, '@id', None)}")
    fields = getattr(rs, 'fields', [])
    for f in fields:
        print(f"    - Field: {getattr(f, 'name', 'no name')} | @id: {getattr(f, '@id', None)}")

Let's inspect the records from the first record set (if present):

In [ ]:
# Choose the first available record set for exploration
if record_sets:
    record_set_id = getattr(record_sets[0], '@id')
    print(f"\nDisplaying 3 records from record set @id: {record_set_id}\n")
    for i, rec in enumerate(dataset.records(record_set=record_set_id)):
        print(rec)
        if i >= 2:
            break
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Extract data from all available record sets using their `@id`, and load them into pandas DataFrames for analysis.

In [ ]:
# Extract all record sets @id in a list
record_set_ids = [getattr(rs, '@id') for rs in record_sets] if record_sets else []

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

if dataframes:
    # Preview the first record set's columns and first few rows
    first_rs_id = record_set_ids[0]
    print(f"Columns in first DataFrame (@id={first_rs_id}):\n{dataframes[first_rs_id].columns.tolist()}\n")
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes were loaded. Check the Croissant schema and data URLs.")

## 4. Exploratory Data Analysis (EDA)
Let's apply common data processing: filter on a numeric field, normalize it, and group by a categorical field—referencing everything by `@id`.

First, we print the columns in our representative DataFrame to help select appropriate fields for analysis.

In [ ]:
# Get representative DataFrame:
if dataframes:
    df = dataframes[first_rs_id]
    print("DataFrame columns are (possible croissant @id's as column names):")
    print(df.columns.tolist())
else:
    print("No dataframe available for EDA.")

Assume the following (adjust these `@id` values matching your output above if needed):
- Numeric field: `age` (referenced by `@id`)
- Grouping field: `sex` (referenced by `@id`)

Replace these with schema-accurate `@id`s as needed. Below is a template; adjust `numeric_field_id` and `group_field_id` to match what is present in your loaded dataframe.

In [ ]:
# Define your numeric and group field @id here based on output above:
numeric_field_id = 'age'  # Replace with e.g., the @id of the age field in your schema
group_field_id = 'sex'    # Replace with e.g., the @id of the sex field in your schema

if numeric_field_id in df.columns:
    # Convert to numeric (robust to possible string storage)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = 50  # Example: filter for age > 50
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (N={filtered_df.shape[0]}):")
    display(filtered_df.head())
    
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group field if available
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print(f"Column {numeric_field_id} not found. Please adjust the field name to match your dataframe columns.")

## 5. Visualization
Visualize distributions and relationships for the selected fields using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the age distribution if the numeric column is present
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group_field
    if group_field_id in df.columns:
        plt.figure(figsize=(7,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
- The FAIR² dataset was successfully loaded and explored using its Croissant schema and the `mlcroissant` library.
- Each record set, field, and column is referenced by its `@id`, ensuring robust and reproducible data operations.
- The EDA showcases how to filter, normalize, and group data using the schema IDs.
- Visualizations provide further insights into distributions and relationships within the data.

For further analysis, refer to the schema to expand on more sophisticated queries or integrate with downstream machine learning workflows.